# Videos → Gaussian splats (gsplat on Colab)

1. **Runtime → Change runtime type → T4 GPU.**
2. In Google Drive, set up this folder (files from the repo's `code/`):
   ```
   MyDrive/loputoo/
   ├── video2splat.py
   ├── runs.py             ← the list of runs (video + settings)
   ├── videos/             ← the videos, straight from the iPhone is fine
   └── prep/               ← optional: frames + COLMAP made on the Mac with code/prep_local.py
   ```
3. **Runtime → Run all.** Keep the tab open; free Colab disconnects idle sessions.

For each run, Colab reuses the matching `prep/` folder if there is one. Otherwise it extracts frames and runs COLMAP itself. Doing it on the Mac is better: Colab's 2 CPU cores are slow, COLMAP uses up GPU time while the GPU sits idle, and Colab's COLMAP (3.7 from apt) has no global mapper. On the first iPhone video, the incremental mapper placed only 2 of 200 frames (tested with COLMAP 4.2.1), while the global mapper placed 196.

What ends up on Drive, per run, in `outputs/<run name>/`: `splat.ply`, `summary.json` (metrics), `renders/` (real vs rendered held-out frames), and the exact settings (`prep.json`, `run.json`, `cfg.yml`).

If the session dies, reconnect and **Run all** again. Finished runs are skipped.

Rough times on a T4 (not measured yet): install ~10 min, training 10–20 min per run at `steps=0.25`, about an hour at `steps=1.0`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# COLMAP from apt (CPU-only build, fine for a few hundred frames), gsplat pinned to v1.5.3.
!apt-get -qq update && apt-get -qq install -y colmap ffmpeg > /dev/null
![ -d /content/gsplat ] || git clone -q --branch v1.5.3 https://github.com/nerfstudio-project/gsplat.git /content/gsplat
!pip install -q gsplat==1.5.3
# examples/requirements.txt pins numpy<2, which fights Colab's own packages; the code works on numpy 2.
!grep -v '^numpy' /content/gsplat/examples/requirements.txt > /content/req.txt
# fused-ssim / fused-bilagrid import torch in setup.py, hence --no-build-isolation.
!pip install -q --no-build-isolation -r /content/req.txt

In [ ]:
# The run list lives in MyDrive/loputoo/runs.py (copy of code/runs.py). To change it, edit
# code/runs.py and copy it over again, or edit it on Drive (Colab's file browser can open it).
DRIVE = "/content/drive/MyDrive/loputoo"
ONLY = ""   # e.g. "short" to run only the runs whose name contains "short"

import importlib, sys
sys.path.insert(0, DRIVE)
import runs as runlist
importlib.reload(runlist)
import os
runs = [r for r in runlist.runs() if ONLY in r["name"]]
names = [r["name"] for r in runs]
for r in runs:
    have = "prep on Drive" if os.path.exists(f"{DRIVE}/prep/{runlist.prep_key(r)}/sparse/0") else "needs COLMAP here"
    print(f"{r['name']:40s} {runlist.prep_key(r):36s} {have}")

In [ ]:
import glob, os, re, shutil, sys
from pathlib import Path
from PIL import Image

missing = sorted({r["video"] for r in runs if not os.path.exists(f"{DRIVE}/videos/{r['video']}")})
assert not missing, f"not found in {DRIVE}/videos/: {missing}"

def sh(args):
    get_ipython().system(" ".join(f'"{a}"' for a in args))

status = {}
for r in runs:
    name = r["name"]
    out = f"{DRIVE}/outputs/{name}"
    if os.path.exists(f"{out}/summary.json"):
        status[name] = "done earlier"
        print(f"=== {name}: already on Drive, skipping")
        continue
    key = runlist.prep_key(r)
    work = f"/content/work/{key}"     # Colab disk: fast, wiped when the session ends
    cache = f"{DRIVE}/prep/{key}"     # frames + COLMAP kept on Drive (from the Mac or an earlier session)
    print(f"\n=== {name}  (frames + COLMAP: {key})")

    # 1. Frames + COLMAP: reuse the Drive copy, or make it here
    if not os.path.exists(work) and os.path.exists(f"{cache}/sparse/0"):
        shutil.copytree(cache, work)
    script = [sys.executable, f"{DRIVE}/video2splat.py", f"{DRIVE}/videos/{r['video']}", work]
    sh(script + runlist.prep_args(r) + ["--stop-after", "sfm"])
    if not os.path.exists(f"{work}/sparse/0/cameras.bin"):
        status[name] = "FAILED in frames / COLMAP (see output above)"
        continue
    if not os.path.exists(cache):   # copy under a temp name so a cut-off copy is never used
        shutil.copytree(work, cache + ".partial", dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("database.db", "runs"))
        os.rename(cache + ".partial", cache)

    # 2. Training
    sh(script + runlist.prep_args(r) + ["--gsplat-dir", "/content/gsplat"] + runlist.train_args(r))
    res = f"{work}/runs/{name}"
    if not os.path.exists(f"{res}/summary.json"):
        status[name] = "FAILED in training (see output above)"
        continue

    # 3. Results to Drive (checkpoints stay behind, they are large)
    os.makedirs(out, exist_ok=True)
    shutil.copy(f"{work}/prep.json", out)
    # Renders: gsplat saves a ~4 MB PNG per held-out frame at every evaluation.
    # Keep only the last evaluation, as JPEG (~1 MB each).
    renders = glob.glob(f"{res}/renders/val_step*.png")
    if renders:
        last = max(int(re.search(r"step(\d+)", p).group(1)) for p in renders)
        os.makedirs(f"{out}/renders", exist_ok=True)
        for p in renders:
            if f"step{last}_" in p:
                Image.open(p).convert("RGB").save(f"{out}/renders/{Path(p).stem}.jpg", quality=90)
    for p in ("splat.ply", "summary.json", "run.json", "cfg.yml", "stats"):
        if os.path.isdir(f"{res}/{p}"):
            shutil.copytree(f"{res}/{p}", f"{out}/{p}", dirs_exist_ok=True)
        elif os.path.exists(f"{res}/{p}"):
            shutil.copy(f"{res}/{p}", out)
    status[name] = "ok"

print("\n=== status")
for name, s in status.items():
    print(f"{name:40s} {s}")

In [ ]:
# Compare the runs. Below the table: for each run, one held-out frame (left = real photo, right = render).
import glob, json, re
import pandas as pd
from IPython.display import Image, display

rows = []
for name in names:
    f = f"{DRIVE}/outputs/{name}/summary.json"
    if os.path.exists(f):
        s = json.load(open(f))
        rows.append({"run": name, **{k: s.get(k) for k in (
            "frames", "registered_frames", "trainer_config", "steps_scaler", "trainer_extra_args",
            "psnr", "ssim", "lpips", "num_gaussians", "train_seconds", "ply_mb")}})
display(pd.DataFrame(rows).set_index("run") if rows else "no finished runs yet")

def step_and_index(p):  # val_step6999_0003.jpg -> (6999, -3): latest step, first frame
    step, i = map(int, re.findall(r"\d+", Path(p).name)[:2])
    return step, -i

for name in names:
    renders = glob.glob(f"{DRIVE}/outputs/{name}/renders/val_step*.jpg")
    if renders:
        print(name)
        display(Image(max(renders, key=step_and_index), width=900))

**Viewing:** download `splat.ply` from `MyDrive/loputoo/outputs/<run name>/` and drag it into
[SuperSplat](https://superspl.at/editor) in the browser on your Mac.